In [0]:
from pyspark.sql import functions as F

vol   = "/Volumes/capstone6_telecom/telecom/ext_raw_landing"
src   = f"{vol}/adls/status_updates/"
table = "capstone6_telecom.telecom.bronze_status_updates"

df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("header", "true")
      .option("cloudFiles.schemaLocation", f"{vol}/_schemas/status_updates")
      .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
      .option("cloudFiles.inferColumnTypes", "true")
      .option("cloudFiles.schemaHints", "source_sequence BIGINT")
      .load(src)
      .withColumn("source_file", F.col("_metadata.file_path"))
      .withColumn("ingest_ts", F.current_timestamp()))

(df.writeStream
   .option("checkpointLocation", f"{vol}/_checkpoints/status_updates")
   .option("mergeSchema", "true")
   .trigger(availableNow=True)
   .toTable(table)
   .awaitTermination())

In [0]:
%sql
SELECT COUNT(*) FROM capstone6_telecom.telecom.bronze_status_updates